# Task 1.1 — Core Contribution

**Selected Paper:** Learning Kernels with Radiuses of Minimum Enclosing Balls (NeurIPS 2010)

Below I walk through the full pipeline of the method proposed in this paper, step by step. The idea is to explain each piece so that someone reading this for the first time can follow the logic from start to finish.

---
## Step 1 — Input Dataset and Classification Task

The setting is straightforward **binary classification**. We have a training set of $n$ data points, each with a label that is either $+1$ or $-1$:

$$D = \{(x_1, y_1), \ldots, (x_n, y_n)\}, \quad x_i \in \mathcal{X},\; y_i \in \{+1, -1\}$$

Now, instead of working directly with the raw inputs, we use a **kernel function** $k$ that implicitly maps every input $x$ into a (possibly very high-dimensional) feature space through a transformation $\phi(\cdot\,; k)$. The neat thing about kernels is that we never need to compute $\phi$ explicitly — we only need the inner products:

$$k(x_c, x_d) = \langle \phi(x_c; k),\; \phi(x_d; k) \rangle$$

Once the data lives in this feature space, we build a linear classifier on top of it (Section 2, Eq. 1):

$$f(x;\, w, b, k) = \langle \phi(x; k),\, w \rangle + b$$

The sign of $f$ tells us the predicted class. The whole point of this paper is to **learn both the kernel $k$ and the classifier $(w, b)$ at the same time**, so that the kernel is actually tailored to our classification task.

---
## Step 2 — Kernel Representation Using Linear Combination of Kernels (Eq. 2)

In practice we usually have several candidate kernels — say, Gaussian RBFs with different bandwidths, polynomial kernels of different degrees, and so on. Instead of picking just one, the paper says: let's **mix them** by taking a weighted sum of $M$ basis kernels $k_1, k_2, \ldots, k_M$:

$$k^{(\theta)}(\cdot,\, \cdot) = \sum_{j=1}^{M} \theta_j \, k_j(\cdot,\, \cdot), \quad \theta_j \geq 0 \tag{Eq. 2}$$

The weights $\theta = (\theta_1, \ldots, \theta_M)$ are called the **combination coefficients**, and they are what we want to learn. A large $\theta_j$ means basis kernel $k_j$ is important; a zero $\theta_j$ means it gets dropped entirely.

This "learn-the-combination" approach is the standard **Multiple Kernel Learning (MKL)** setup. So the learning task boils down to: find the best weights $\theta$ and, at the same time, train the SVM on the resulting combined kernel.

---
## Step 3 — Problems with Margin-Based MKL (Section 2.1)

The usual way to do MKL is to use the **large margin principle** — basically, find the kernel + classifier combination that maximises the SVM margin. The classical formulation looks like this (Eq. 3–5):

$$\min_{k \in \mathcal{K}} \;\widetilde{G}(k) = \min_{w,b,\xi_i} \; \frac{1}{2}\|w\|^2 + C \sum_i \xi_i$$
$$\text{s.t.} \quad y_i(\langle \phi(x_i; k),\, w \rangle + b) + \xi_i \geq 1, \quad \xi_i \geq 0$$

This looks reasonable, but the authors show it has **two serious flaws**:

### 3a. The Scaling Problem

Here's the issue. Suppose we just multiply our kernel by some big number $a > 1$, so $k_{\text{new}} = a \cdot k$. The feature map then scales as $\phi(\cdot\,; k_{\text{new}}) = \sqrt{a}\,\phi(\cdot\,; k)$. If we also set $w_2 = w_1^*/\sqrt{a}$, the decision boundary doesn't change at all — we're classifying points exactly the same way — but the norm of $w$ shrinks:

$$\|w_2\|^2 = \frac{\|w_1^*\|^2}{a}$$

So the objective value goes down: $\widetilde{G}(a k) < \widetilde{G}(k)$. In other words, **we can make any kernel look great just by making it bigger**, even a completely useless kernel. The margin is being "tricked" by the scale, so it's not a fair measure of how good a kernel really is.

### 3b. The Initialisation Problem

You might think: "OK, let's just add a constraint like $\theta_1 + \theta_2 = 1$ to keep the scale under control." That helps, but it doesn't fully fix things. The authors give a nice example (Section 2.1, Eq. 6): imagine two basis kernels $k_1$ and $k_2$, where $k_1$ happens to give a larger margin. MKL will pick $k_1$. But now, if someone rescales $k_1$ by a small factor (making it $k_1' = a \cdot k_1$ with $a$ small enough), suddenly $k_2$ has the larger margin and MKL flips to $k_2$. Nothing about the actual geometry of the data changed — only the initial scaling of the basis kernels did.

**The bottom line:** The margin on its own is tangled up with the kernel's scale, so it can't reliably tell us which kernel is actually better for our data.

---
## Step 4 — Margin-Radius Principle Using Minimum Enclosing Ball (Section 2.2)

So if the margin by itself is unreliable, what should we use instead? The authors rely on **generalisation bounds** for SVMs. These bounds (Section 2.2, citing [14, 15]) tell us that the expected error is controlled not just by the margin, but by the **ratio** of the margin to the size of the data cloud:

$$\mathbb{E}[\text{error}] \leq \frac{\mathbb{E}[R^2 / \gamma^2]}{n}$$

Here $\gamma = 1/\|w\|$ is the margin and $R$ is the **radius of the minimum enclosing ball (MEB)** — basically the radius of the smallest sphere in feature space that contains all training points. The key insight is that $R^2 \|w\|^2$ naturally cancels out any scaling of the kernel (because $R^2$ scales up by $a$ while $\|w\|^2$ scales down by $1/a$), so this ratio is a much fairer way to judge a kernel.

### What exactly is the Minimum Enclosing Ball?

Given a kernel $k$, the squared MEB radius $R^2(k)$ is the solution to a simple optimisation problem. In **primal form** (Eq. 7), we're looking for the smallest sphere centred at $c$ that covers all data points:

$$R^2(k) = \min_{y, c} \; y \quad \text{s.t.} \quad y \geq \|\phi(x_i; k) - c\|^2 \quad \forall i$$

Equivalently, the **dual form** (Eq. 8) lets us compute it using only kernel evaluations:

$$R^2(k) = \max_{\beta} \; \sum_i \beta_i \, k(x_i, x_i) - \sum_{i,j} \beta_i \, k(x_i, x_j) \, \beta_j$$
$$\text{s.t.} \quad \sum_i \beta_i = 1, \quad \beta_i \geq 0$$

A really important property is **positive homogeneity**: $R^2(a k) = a \cdot R^2(k)$. You can see this directly from the dual — every term in the objective is linear in $k$. This scaling behaviour is exactly what makes the margin-radius ratio invariant, and it's the foundation of the proposed method.

---
## Step 5 — Radius-Based Kernel Learning Objective (RKL, Eq. 9)

Now we get to the main contribution. The authors propose a new formulation they call **RKL (Radius-based Kernel Learning)**. The idea is simple: instead of regularising with just $\|w\|^2$ (which only looks at the margin), multiply it by $R^2(k)$ so the objective reflects the margin-to-radius ratio:

$$\min_{k, w, b, \xi_i} \; \frac{1}{2} R^2(k) \|w\|^2 + C \sum_i \xi_i \tag{Eq. 9}$$
$$\text{s.t.} \quad y_i(\langle \phi(x_i; k),\, w \rangle + b) + \xi_i \geq 1, \quad \xi_i \geq 0$$

### What changed compared to standard SVM?

- The old regulariser was $\frac{1}{2}\|w\|^2$. The new one is $\frac{1}{2} R^2(k) \|w\|^2$. We're penalising not just a small margin, but a small margin **relative to how spread out the data is** in the kernel's feature space.
- This means the objective now prefers kernels where the ratio $\gamma / R$ (margin over radius) is large, which is exactly what the generalisation bounds suggest.

### Why this fixes the scaling problem (Proposition 1)

Let $G(k)$ be the optimal objective value for a given kernel $k$. The paper proves:

$$G(ak) = G(k) \quad \forall\, a > 0 \tag{Proposition 1}$$

The reason is intuitive: scaling the kernel by $a$ makes $R^2$ grow by $a$, but the optimal $\|w\|^2$ shrinks by $1/a$ (because the classifier adjusts). The two effects cancel out perfectly, so the objective value doesn't change. You can no longer "cheat" by inflating the kernel.

### Bonus invariance properties

The paper also shows two more results that follow from this scaling invariance:
- **Proposition 3 (basis kernel scaling):** If someone rescales individual basis kernels by arbitrary positive factors $a_j$ (say, $k_1 \to 5 k_1$, $k_2 \to 0.1 k_2$), the optimal solution of RKL doesn't change. This kills the initialisation problem.
- **Proposition 2 (norm constraint type):** Whether we constrain $\theta$ with an L1 norm, L2 norm, or no norm at all, we get equivalent solutions. So there's no need to worry about which constraint to pick.

Together, these results mean RKL completely addresses both the scaling and initialisation problems from Section 2.1.

---
## Step 6 — Tri-Level Optimisation Formulation (Section 3.2)

OK so the RKL objective looks clean, but how do we actually solve it? For the linear combination case $k^{(\theta)} = \sum_j \theta_j k_j$, the authors reformulate it as a **tri-level (three-nested) optimisation problem** (Eqs. 14–17). Think of it as three problems stacked inside each other:

### Top level — choosing the kernel weights $\theta$:
$$\min_{\theta} \; g(\theta) \quad \text{s.t.} \quad \theta_j \geq 0 \tag{Eq. 14}$$

This is what we ultimately want to minimise. But to evaluate $g(\theta)$ for a given $\theta$, we need to solve the middle level first.

### Middle level — solving an SVM dual:
$$g(\theta) = \max_{\alpha} \; \sum_i \alpha_i - \frac{1}{2 r^2(\theta)} \sum_{i,j} \alpha_i y_i K_{ij}(\theta) y_j \alpha_j \tag{Eq. 16}$$
$$\text{s.t.} \quad 0 \leq \alpha_i \leq C, \quad \sum_i \alpha_i y_i = 0$$

where $K_{ij}(\theta) = \sum_j \theta_j K_j$ is the combined kernel matrix. Notice that $r^2(\theta)$ appears in the objective — this is the squared MEB radius, which itself comes from yet another optimisation:

### Bottom level — computing the MEB radius:
$$r^2(\theta) = \max_{\beta} \; \sum_i \beta_i K_{ii}(\theta) - \sum_{i,j} \beta_i K_{ij}(\theta) \beta_j \tag{Eq. 17}$$
$$\text{s.t.} \quad \sum_i \beta_i = 1, \quad \beta_i \geq 0$$

So the flow goes: **bottom level** computes the radius $r^2(\theta)$ → **middle level** plugs that radius into the SVM and solves for $\alpha^*$ and the objective $g(\theta)$ → **top level** adjusts $\theta$ to make $g(\theta)$ as small as possible. This nesting is what makes the problem trickier than standard MKL.

---
## Step 7 — Gradient Projection Optimisation Algorithm (Section 5)

To actually solve this tri-level problem, the authors use a **gradient projection** approach. The crucial enabler is that they prove $g(\theta)$ is **continuously differentiable** — this is done by generalising Danskin's theorem to multi-level optimisation problems (Theorem 1, Section 4). Once we know gradients exist and are smooth, we can just do gradient descent on $\theta$.

### How the gradient is computed (Eq. 21)

The gradient with respect to each kernel weight $\theta_l$ turns out to be:

$$\frac{\partial g}{\partial \theta_l} = \frac{1}{2r^2} \left( \sum_{i,j} \alpha_i^* y_i (K_l)_{ij} y_j \alpha_j^* \right) \left( \frac{\sum_{i,j} \beta_i^* (K_l)_{ij} \beta_j^* - \sum_i \beta_i^* (K_l)_{ii}}{r^2} \right) - \frac{1}{2r^2} \sum_{i,j} \alpha_i^* y_i (K_l)_{ij} y_j \alpha_j^*$$

This looks complicated, but it's built entirely from things we already computed: $\alpha_i^*$ (from the SVM dual), $\beta_i^*$ (from the MEB dual), and the individual kernel matrices $K_l$. So once we've solved those two QPs, getting the gradient is cheap.

### The algorithm, step by step:

1. **Start** with some initial $\theta$ — for example, equal weights $\theta_j = 1/M$.
2. **Bottom level:** Solve the MEB dual (Eq. 17) to get $\beta^*$ and the radius $r^2(\theta)$.
3. **Middle level:** Solve the SVM dual (Eq. 16), plugging in $r^2(\theta)$, to get $\alpha^*$ and the objective $g(\theta)$.
4. **Gradient:** Compute $\nabla_\theta g$ using the formula above (Eq. 21).
5. **Update:** Take a gradient step $\theta \leftarrow \theta - \eta \nabla_\theta g$.
6. **Project:** Push $\theta$ back onto the feasible set (so all $\theta_j \geq 0$ and any norm constraint is satisfied):
   - For **L1** ($\sum_j \theta_j = 1$, $\theta_j \geq 0$): use the efficient projection of Duchi et al. [19].
   - For **L2** ($\sum_j \theta_j^2 = 1$, $\theta_j \geq 0$): clip negatives to zero, then normalise.
   - For **no constraint** ($\theta_j \geq 0$): just clip negatives to zero.
7. **Repeat** steps 2–6 until convergence.

### Why this is practical:
- The SVM and MEB subproblems both have the form of a QP with box constraints, so they can be solved with fast **SMO-style solvers**.
- Because $\alpha^*$ and $\beta^*$ change smoothly as $\theta$ changes (Theorem 1 guarantees this), we can **warm-start** each iteration with the previous solution, which makes each solve much faster.
- In the experiments, the whole thing typically converges within **10–20 outer iterations** (Section 5), which is very reasonable.
- One caveat: the RKL problem is **non-convex**, so we only get a local optimum. But Proposition 4 provides reassurance — it shows that every local optimum of RKL is actually the global optimum of a certain related convex problem (Eq. 23), so local optima are still meaningful.

---
## Step 8 — Final Learned Kernel and Classifier

Once the algorithm converges, here's what we end up with:

1. **Learned kernel weights** $\theta^* = (\theta_1^*, \ldots, \theta_M^*)$. These tell us the final combined kernel:
$$k^{(\theta^*)}(\cdot,\, \cdot) = \sum_{j=1}^{M} \theta_j^* \, k_j(\cdot,\, \cdot)$$
Many of the $\theta_j^*$ will be zero or near-zero, so the method effectively **selects a small subset** of the basis kernels.

2. **SVM support vector coefficients** $\alpha^*$, which define the classifier weight vector:
$$w^* = \sum_i \alpha_i^* y_i \, \phi(x_i;\, k^{(\theta^*)})$$

3. **Bias term** $b^*$, computed from the KKT conditions of the SVM dual.

4. To **classify a new test point** $x$, we compute:
$$\hat{y} = \text{sign}\left( \sum_i \alpha_i^* y_i \, k^{(\theta^*)}(x_i, x) + b^* \right)$$
Notice we never need to compute $\phi$ explicitly — everything goes through kernel evaluations, as usual with SVMs.

### How well does it work?

In the experiments (Table 1, Section 6), RKL consistently outperforms:
- **Unif** — SVM with equal weights on all basis kernels (no learning),
- **MKL** — standard margin-based multiple kernel learning,
- **KL-C** — the radius-aware method of Chapelle et al. [1],

across 11 benchmark datasets. The improvements can be quite large (e.g., 5–10% accuracy gains on datasets like Splice and Musk1). The method also tends to pick **sparse combinations** (only a few basis kernels get nonzero weight), and — as predicted by Proposition 2 — the results are **the same regardless of whether we use an L1, L2, or no norm constraint** on $\theta$.

---
## Summary

The paper solves the **scaling and initialisation problems** in margin-based Multiple Kernel Learning, where the learned kernel is biased by arbitrary kernel magnitudes rather than intrinsic data geometry. The proposed Radius-based Kernel Learning (RKL) approach is better because its objective $R^2(k)\|w\|^2$ is provably invariant to kernel scaling, basis kernel scaling, and the choice of norm constraint, leading to more reliable kernel selection and significantly improved classification accuracy.